# OR Capacity Model: Baseline vs Volume Projections

Python port of `model/or_model.R` (plus the helpers it uses from
`functions/or_constants_functions.R`).

**What it does**
1. Loads the baseline and scenario inpatient volume (`new_daily_demand_encounter.xlsx`).
2. Pulls completed OR cases from Oracle (`MS_INSIGHT`) and tags the ones that fall inside a baseline IP stay (the cohort).
3. Builds one surgery "template" per cohort patient and clones it onto every new (dummy) encounter in the scenario.
4. Computes prime-time metrics by hospital and year for Baseline and Volume Projections, with deltas.
5. Computes hourly room demand against staffed capacity (cascade schedule) and saves one plot per site
   (`demand_vs_staffed_<version>_<site>.png`).
6. Writes the scenario results workbook with a two-row banded header.

**Parity notes.** The port keeps the R script's behavior, including a few quirks that are marked
`# PARITY NOTE` in the code. The main ones:
- `Delta TAT` subtracts Volume Projections from itself, so it is always 0 (same as R).
- Interval overlaps follow lubridate's `intersect()`, which flips reversed intervals. When a room's last case
  runs past prime-time end, the "end gap" becomes the overrun window and is counted as a gap (same as R).
- `get_valid_date()` shifts each date on its own. In R it is called on a whole vector inside `||`, so only the
  first element decided the shift. The shift is random either way, so the dummy surgery dates will not match R
  exactly. Set `RANDOM_SEED` to make runs repeatable.
- Timestamps are naive New York clock times, the same as `force_tz(..., "America/New_York")` in R.

## Setup

In [ ]:
# On Databricks, uncomment if these aren't on the cluster:
# %pip install openpyxl holidays
# dbutils.library.restartPython()

In [ ]:
import os
import sys
from datetime import datetime, time as dt_time, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import holidays as pyholidays
from dateutil.easter import easter
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## Configuration

In [ ]:
# Root folder that holds archive/, OR Modeling/SupplementData/ and OR Modeling/Outputs/
file_location = "/SharedDrive/deans/Presidents/HSPI-PM/Operations Analytics and Optimization/Projects/System Operations/Capacity Modeling/"

# Oracle connection helper used by the other Databricks notebooks in this repo
DB_CONFIG_PATH = "/Volumes/opsanalytics_adb_workspace01/default/oracle_connections"
DB_ENVIRONMENT = "production"

VOLUME_FILE_NAME = "new_daily_demand_encounter.xlsx"
CASCADE_SHEET = "10-2025"
PLOT_VERSION = "v9"          # plots are saved as demand_vs_staffed_<PLOT_VERSION>_<site>.png
EXCLUDE_YEAR = 2026          # demand rows for this year are dropped before plotting
RANDOM_SEED = 2026           # seed for the random holiday/weekend shift of dummy surgery dates
RECOVERABLE_THRESHOLD = 180  # minutes; gaps at or above this are "recoverable"

plot_dir = os.path.join(file_location, "OR Modeling/Outputs/DemandPlots/")
output_dir = os.path.join(file_location, "OR Modeling/Outputs/")
os.makedirs(plot_dir, exist_ok=True)

rng = np.random.default_rng(RANDOM_SEED)

## Constants and helpers
(from `functions/or_constants_functions.R`)

In [ ]:
# MSHS brand palette
mshs_cyan    = "#06ABEB"
mshs_magenta = "#DC298D"
mshs_violet  = "#212070"
mshs_gray    = "#63666A"

# Location crosswalk: OR LOCATION_NAME -> cascade Location code (first match wins)
loc_xwalk = [
    ("5 E 98TH ST AMB SUITE", "MSH"),
    ("MSDC|MSDUS",            "MSBI"),
    ("MSH",                   "MSH"),
    ("MSW",                   "MSW"),
    ("MSM",                   "MSM"),
    ("MSQ",                   "MSQ"),
    ("MSBI",                  "MSBI"),
    ("MSB",                   "MSB"),
    ("MSSN",                  "MSSN"),
]


def xwalk_loc(loc):
    """Map OR LOCATION_NAME values to cascade Location codes (NaN when nothing matches)."""
    loc = pd.Series(loc).astype("string")
    out = pd.Series(np.nan, index=loc.index, dtype="object")
    for pattern, code in loc_xwalk:
        hit = out.isna() & loc.str.contains(pattern, regex=True, na=False)
        out[hit] = code
    return out


# MSHS holidays = NYSE holidays without Good Friday, plus Black Friday
def _thanksgiving(year):
    nov1 = pd.Timestamp(year=year, month=11, day=1)
    return nov1 + pd.Timedelta(days=(3 - nov1.weekday()) % 7 + 21)


_years = range(1990, 2101)
_nyse = {pd.Timestamp(d) for d in pyholidays.NYSE(years=_years).keys()}
_good_friday = {pd.Timestamp(easter(y)) - pd.Timedelta(days=2) for y in _years}
_black_friday = {_thanksgiving(y) + pd.Timedelta(days=1) for y in _years}
mshs_holiday = sorted((_nyse - _good_friday) | _black_friday)
mshs_holiday_set = set(mshs_holiday)


def is_invalid_day(dates):
    """True for holidays, Saturdays and Sundays."""
    dates = pd.to_datetime(pd.Series(dates))
    return dates.isin(mshs_holiday_set) | (dates.dt.dayofweek >= 5)


def get_valid_date(start_dates, holidays_set=mshs_holiday_set, rng=rng):
    """Move holiday/weekend dates forward by a random 1-5 days until they land on a working day.

    PARITY NOTE: applied row by row. See the note at the top of the notebook.
    """
    dates = pd.to_datetime(pd.Series(start_dates)).dt.normalize()
    out = dates.copy()
    for idx in dates.index[is_invalid_day(dates).to_numpy() & dates.notna().to_numpy()]:
        start = dates[idx]
        new_date = start
        while new_date in holidays_set or new_date.dayofweek >= 5:
            new_date = start + pd.Timedelta(days=int(rng.integers(1, 6)))
        out[idx] = new_date
    return out


def parse_clock(x):
    """Parse a schedule time ("7:00:00 AM", "07:00", datetime.time, Excel datetime) into a time-of-day Timedelta."""
    if x is None or (isinstance(x, float) and np.isnan(x)) or x is pd.NaT:
        return pd.NaT
    if isinstance(x, (datetime, pd.Timestamp)):
        return pd.Timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
    if isinstance(x, dt_time):
        return pd.Timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
    if isinstance(x, (int, float)):  # Excel fraction of a day
        return pd.Timedelta(days=float(x)).round("s")
    s = str(x).strip()
    for fmt in ("%I:%M:%S %p", "%I:%M %p", "%H:%M:%S", "%H:%M"):
        try:
            t = datetime.strptime(s, fmt)
            return pd.Timedelta(hours=t.hour, minutes=t.minute, seconds=t.second)
        except ValueError:
            continue
    raise ValueError(f"Unrecognized time value: {x!r}")


def _std_interval(start, end):
    """lubridate::int_standardize: flip intervals whose end is before their start."""
    flip = end < start
    return start.where(~flip, end), end.where(~flip, start)


def overlap_minutes(a_start, a_end, b_start, b_end):
    """Length in minutes of lubridate::intersect(interval(a), interval(b)).

    NaN when either interval is missing or they don't overlap (NA interval in R).
    Reversed intervals are flipped first, as lubridate does.
    """
    a_start, a_end = _std_interval(pd.Series(a_start), pd.Series(a_end))
    b_start, b_end = _std_interval(pd.Series(b_start), pd.Series(b_end))
    start = a_start.where(a_start >= b_start, b_start)
    end = a_end.where(a_end <= b_end, b_end)
    span = (end - start).dt.total_seconds() / 60
    missing = a_start.isna() | a_end.isna() | b_start.isna() | b_end.isna()
    return span.mask(missing | (span < 0))


def add_minutes(ts, minutes):
    """ts + minutes(as.integer(minutes)); NaT when minutes is missing."""
    return ts + pd.to_timedelta(np.trunc(pd.to_numeric(minutes, errors="coerce")), unit="m")


def explode_hours(df, start_col, end_col, out_col="hour_ts"):
    """One row per clock hour from floor(start) to floor(end), inclusive (seq(..., by = "hour"))."""
    df = df.reset_index(drop=True)
    h0 = df[start_col].dt.floor("h")
    h1 = df[end_col].dt.floor("h")
    n = ((h1 - h0) / pd.Timedelta(hours=1)).fillna(-1).astype(int) + 1
    keep = n > 0
    df, h0, n = df[keep], h0[keep], n[keep]
    out = df.loc[df.index.repeat(n)].copy()
    step = out.groupby(level=0).cumcount().to_numpy()
    out[out_col] = h0.loc[out.index].to_numpy() + pd.to_timedelta(step, unit="h")
    return out.reset_index(drop=True)


def as_id_string(s):
    """Like paste0() on an id column: 123.0 -> "123"."""
    def conv(v):
        if pd.isna(v):
            return np.nan
        if isinstance(v, float) and v.is_integer():
            return str(int(v))
        return str(v)
    return pd.Series(s).map(conv)

### Cascade staffing schedule
Rooms staffed per location and time band, from `ORSchedules.xlsx`.
`factor` = rooms open in the band / peak rooms for the location.

In [ ]:
cascade_factor = pd.read_excel(
    os.path.join(file_location, "OR Modeling/SupplementData/ORSchedules.xlsx"),
    sheet_name=CASCADE_SHEET,
)
cascade_factor = cascade_factor[cascade_factor["Location"].notna()
                                & (cascade_factor["Location"].astype(str).str.strip() != "")].copy()
cascade_factor["start_td"] = cascade_factor["Time Start"].map(parse_clock)
cascade_factor["end_td"] = cascade_factor["Time End"].map(parse_clock)
cascade_factor["peak"] = cascade_factor.groupby("Location")["# ORs"].transform("max")
cascade_factor["factor"] = cascade_factor["# ORs"] / cascade_factor["peak"]
cascade_factor["band_min"] = (cascade_factor["end_td"] - cascade_factor["start_td"]).dt.total_seconds() / 60
cascade_factor["capacity_min"] = cascade_factor["# ORs"] * cascade_factor["band_min"]
cascade_factor = cascade_factor[["Location", "Time Start", "Time End", "start_td", "end_td",
                                 "# ORs", "factor", "capacity_min"]].reset_index(drop=True)
cascade_factor

In [ ]:
def cascade_hourly_capacity(cascade=cascade_factor):
    """Staffed rooms per Location x hour of day. Each band covers floor(start)..floor(end - 1 min)."""
    cap = cascade[["Location", "start_td", "end_td", "# ORs"]].copy()
    day = pd.Timestamp("2000-01-01")
    cap["START"] = day + cap["start_td"]
    cap["END"] = day + cap["end_td"] - pd.Timedelta(minutes=1)
    cap = cap.rename(columns={"# ORs": "Capacity"})
    cap = explode_hours(cap, "START", "END", out_col="hour_ts")
    cap["hour_surgery"] = cap["hour_ts"].dt.hour
    return cap[["Location", "hour_surgery", "Capacity"]]

### Prime time by site

In [ ]:
def prime_time_location(data):
    data = data.copy()
    surgery_date = pd.to_datetime(data["SURGERY_DATE"]).dt.normalize()
    data["Weekday"] = surgery_date.dt.day_name()
    loc = data["LOCATION_NAME"].astype("string")

    def has(p):
        return loc.str.contains(p, regex=False, na=False).to_numpy()

    site = np.select(
        [has("MSDUS"), has("MSDC"), has("MSW"), has("MSH"),
         has("5 E 98TH ST AMB SUITE"), has("MSM"), has("MSQ"), has("MSB")],
        ["MSDUS", "MSDC", "MSW", "MSH", "MSH", "MSM", "MSQ", "MSB"],
        default=None,
    )
    is_wed = (data["Weekday"] == "Wednesday").to_numpy()
    in_dc = np.isin(site, ["MSDC", "MSDUS", "MSW"])
    start_time = np.select(
        [(site == "MSH") & ~is_wed, (site == "MSH") & is_wed,
         (site == "MSM") & is_wed, (site == "MSM") & ~is_wed,
         in_dc & is_wed, in_dc & ~is_wed,
         np.isin(site, ["MSQ", "MSB"])],
        ["08:00:00", "09:00:00", "07:30:00", "08:30:00", "08:30:00", "07:30:00", "08:00:00"],
        default=None,
    )
    end_time = np.select(
        [np.isin(site, ["MSH", "MSQ"]), np.isin(site, ["MSM", "MSDC", "MSDUS", "MSW"]), site == "MSB"],
        ["18:00:00", "17:30:00", "15:00:00"],
        default=None,
    )
    data["PRIME_TIME_START"] = surgery_date + pd.to_timedelta(pd.Series(start_time, index=data.index))
    data["PRIME_TIME_END"] = surgery_date + pd.to_timedelta(pd.Series(end_time, index=data.index))
    return data

### Pull OR cases from Oracle

In [ ]:
OR_CASE_QUERY = """
SELECT
    e.OR_CASE_ID,
    ENCOUNTER_NO_SRC, ENCOUNTER_ID,
    PAYOR_GROUP_DESC_MSX_OP, CCM_PAYOR_DESC_MSX_OP, PAYOR_GROUP_OP,
    CLINIC_GROUP_DESC_MSX, REG_AREA_DESC_SRC,
    ATTENDING_MD_NAME_MSX       AS ATTENDING_MD,
    ATTENDING_MD_SPEC_SRC       AS ATTENDING_MD_SPECIALIZATION,
    PRIMARY_SURGEON,
    SURGEON_SPECIALTY           AS PRIMARY_SURGEON_SPECIALTY,
    PAT_MRN_ID                  AS PATIENT_MRN,
    PAT_DOB                     AS PATIENT_DOB,
    ADMIT_CSN_ID, TOTAL_TIME_NEEDED,
    PRIMARY_PROC_CODE           AS PRIMARY_PROCEDURE_CODE,
    PRIMARY_PROCEDURE           AS PRIMARY_PROCEDURE_DESC,
    ANESTHESIA_TYPE, PRIME_TIME_PROC,
    SCHED_IN_ROOM_DTTM, SCHED_START_TIME,
    PATIENT_IN_ROOM_DTTM, PATIENT_OUT_ROOM_DTTM,
    MINUTES_IN_ROOM_TO_OUT_ROOM, TURNOVER_FROM_PRIOR_CASE,
    SURGERY_DATE,
    OR_ID                       AS ROOM_ID,
    OR_LOCATION                 AS LOCATION_NAME,
    CLUSTER_NAME, ROBOTIC_SURGERY_DAVINCI_YN,
    HOLIDAY_YN, WEEKEND_YN
FROM MS_INSIGHT.OR_QUALITY_DASHBOARD_CASE_DETAILS e
LEFT JOIN MS_INSIGHT.OR_QUALITY_ROOM_UTIL_V u
    ON e.OR_CASE_ID = u.LOG_ID
WHERE SURGERY_DATE >= TO_DATE(?, 'YYYY-MM-DD')
  AND SURGERY_DATE <= TO_DATE(?, 'YYYY-MM-DD')
  AND WEEKEND_YN = 'N'
  AND HOLIDAY_YN = 'N'
  AND OR_LOCATION NOT LIKE 'L&D%'
  AND CASE_STATUS = ?
"""


def fetch_or_cases(sched_start_date, sched_end_date, status):
    if DB_CONFIG_PATH not in sys.path:
        sys.path.append(DB_CONFIG_PATH)
    from db_config import get_connection

    conn = get_connection(DB_ENVIRONMENT)
    try:
        cursor = conn.cursor()
        cursor.execute(OR_CASE_QUERY, [sched_start_date, sched_end_date, status])
        columns = [desc[0] for desc in cursor.description]
        df = pd.DataFrame.from_records(cursor.fetchall(), columns=columns)
    finally:
        conn.close()
    for col in ["SCHED_IN_ROOM_DTTM", "PATIENT_IN_ROOM_DTTM", "PATIENT_OUT_ROOM_DTTM", "SURGERY_DATE"]:
        df[col] = pd.to_datetime(df[col])
    df["PATIENT_MRN"] = df["PATIENT_MRN"].astype("string")
    return df


def get_or_data(sched_start_date="2025-01-01", sched_end_date="2025-12-31", status="Completed", mrn_list=None):
    schedule_data = fetch_or_cases(sched_start_date, sched_end_date, status)
    schedule_data = prime_time_location(schedule_data).drop_duplicates("OR_CASE_ID", keep="first")

    # Attach the IP stay the surgery falls in (latest admit if several)
    stays = mrn_list.rename(columns={"MSMRN": "PATIENT_MRN"})
    ip = schedule_data.merge(stays, on="PATIENT_MRN", how="left")
    in_stay = (ip["SURGERY_DATE"] >= ip["ADMIT_DT_SRC"]) & (ip["SURGERY_DATE"] <= ip["DSCH_DT_SRC"])
    ip.loc[~in_stay, ["ADMIT_DT_SRC", "DSCH_DT_SRC"]] = pd.NaT
    ip = (ip.sort_values(["OR_CASE_ID", "ADMIT_DT_SRC"], ascending=[True, False], na_position="last")
            .drop_duplicates("OR_CASE_ID", keep="first")
            .reset_index(drop=True))

    # Drop holidays, weekends and Dec 24/26/31
    ip["SURGERY_DATE"] = ip["SURGERY_DATE"].dt.normalize()
    sd = ip["SURGERY_DATE"]
    ip = ip[~is_invalid_day(sd).to_numpy()
            & ~((sd.dt.month == 12) & sd.dt.day.isin([24, 26, 31])).to_numpy()].copy()

    # Setup/cleanup after a case = turnover recorded on the next case in the same room
    ip = ip.sort_values(["ROOM_ID", "PATIENT_OUT_ROOM_DTTM"], na_position="last", kind="mergesort")
    ip["setup_and_cleanup_time"] = (ip.groupby("ROOM_ID", dropna=False)["TURNOVER_FROM_PRIOR_CASE"]
                                      .shift(-1).fillna(0))
    ip = ip.reset_index(drop=True)

    ip["PATIENT_OUT_AND_SETUP_CLEANUP_END"] = add_minutes(ip["PATIENT_OUT_ROOM_DTTM"], ip["setup_and_cleanup_time"])
    # Prime-time overlap minutes (NaN = no overlap), i.e. int_length(intersect(...)) / 60
    ip["overlap_primetime_procedure"] = overlap_minutes(ip["PRIME_TIME_START"], ip["PRIME_TIME_END"],
                                                        ip["PATIENT_IN_ROOM_DTTM"], ip["PATIENT_OUT_ROOM_DTTM"])
    ip["overlap_primetime_setup"] = overlap_minutes(ip["PRIME_TIME_START"], ip["PRIME_TIME_END"],
                                                    ip["PATIENT_OUT_ROOM_DTTM"], ip["PATIENT_OUT_AND_SETUP_CLEANUP_END"])

    sd = ip["SURGERY_DATE"]
    ip["month"] = sd.dt.month
    ip["month_date"] = sd.dt.to_period("M").dt.to_timestamp()
    ip["day_of_month"] = sd.dt.day
    ip["week_of_year"] = (sd.dt.dayofyear - 1) // 7 + 1   # lubridate::week()
    ip["year"] = sd.dt.year
    return ip

### Metrics engine
Output grain: Hospital x Year. Used, Procedure and TAT are raw prime-time minutes (daily mean).
Available time and end-of-day gaps are weighted by the cascade staffing curve.
Utilization = raw used / cascade-weighted available, so it can go above 100%.

In [ ]:
def _label_sum(df, keys, value, prefix):
    """Split a gap value into Recoverable / Non Recoverable (>= threshold) and sum per key."""
    out = df[keys].copy()
    v = df[value]
    out[f"Recoverable {prefix}"] = v.where(v >= RECOVERABLE_THRESHOLD, 0).fillna(0)
    out[f"Non Recoverable {prefix}"] = v.where(v < RECOVERABLE_THRESHOLD, 0).fillna(0)
    return out.groupby(keys, dropna=False, as_index=False).sum()


def _with_cascade_window(df):
    """Join the cascade bands by Location and build each band's clock window on SURGERY_DATE."""
    df = df.merge(cascade_factor, on="Location", how="left")
    df["cascade_start"] = df["SURGERY_DATE"] + df["start_td"]
    df["cascade_end"] = df["SURGERY_DATE"] + df["end_td"]
    return df


METRIC_COLS = ["# Cases", "Prime Time Utilization", "Prime Time Used Time", "Prime Time Procedure Time",
               "Prime Time TAT", "Prime Time Available Time", "Prime Time Recoverable Time",
               "Prime Time Non Recoverable Time"]


def summary_metrics_weighted(processed_or_cases, scenario_label):
    base = processed_or_cases[processed_or_cases["overlap_primetime_procedure"].notna()
                              | processed_or_cases["overlap_primetime_setup"].notna()].copy()
    base["Location"] = xwalk_loc(base["LOCATION_NAME"]).to_numpy()

    # --- raw metrics: total per location-day, then mean across days ---
    raw = base.assign(
        proc_pt_min=base["overlap_primetime_procedure"],
        tat_pt_min=base["overlap_primetime_setup"],
        used_pt_min=base["overlap_primetime_procedure"].fillna(0) + base["overlap_primetime_setup"].fillna(0),
        Year=base["SURGERY_DATE"].dt.year,
    )
    raw_day = raw.groupby(["Location", "Year", "SURGERY_DATE"], dropna=False).agg(
        cases_day=("OR_CASE_ID", "size"), used_day=("used_pt_min", "sum"),
        proc_day=("proc_pt_min", "sum"), tat_day=("tat_pt_min", "sum")).reset_index()
    raw_all = raw_day.groupby(["Location", "Year"], dropna=False).agg(**{
        "# Cases": ("cases_day", "sum"),
        "Prime Time Used Time": ("used_day", "mean"),
        "Prime Time Procedure Time": ("proc_day", "mean"),
        "Prime Time TAT": ("tat_day", "mean")}).reset_index()

    # --- gaps (room-assigned cases only) ---
    base_rooms = base[base["ROOM_ID"].notna()].copy()
    rd = ["ROOM_ID", "SURGERY_DATE", "Location"]
    oc = base_rooms.sort_values(rd + ["PATIENT_IN_ROOM_DTTM"], na_position="last", kind="mergesort")
    g = oc.groupby(rd, dropna=False, sort=False)
    oc["case_order"] = g.cumcount() + 1
    oc["first_case_start"] = g["PATIENT_IN_ROOM_DTTM"].transform("min")
    oc["last_case_end_tat_incl"] = g["PATIENT_OUT_AND_SETUP_CLEANUP_END"].transform("max")
    oc["previous_case_end"] = g["PATIENT_OUT_AND_SETUP_CLEANUP_END"].shift(1)

    def pos_minutes(later, earlier):
        d = (later - earlier).dt.total_seconds() / 60
        return d.where(d > 0, 0).mask(d.isna())

    oc["gap_at_start"] = pos_minutes(oc["first_case_start"], oc["PRIME_TIME_START"])
    oc["gap_between"] = pos_minutes(oc["PATIENT_IN_ROOM_DTTM"], oc["previous_case_end"])

    # start-of-day gap: one per room-day
    gs = oc[rd + ["PRIME_TIME_START", "first_case_start", "gap_at_start"]].drop_duplicates()
    gaps_start = _label_sum(gs, rd, "gap_at_start", "Start")

    # between-case gaps: case 2 onward
    gm = (oc.loc[oc["case_order"] > 1, rd + ["case_order", "gap_between"]]
            .drop_duplicates().dropna())
    gaps_mid = _label_sum(gm, rd, "gap_between", "Mid")

    # end-of-day gap, split across cascade bands and weighted by each band's factor
    # PARITY NOTE: overlap_minutes flips a reversed interval like lubridate, so when the last case ends
    # after prime-time end, the overrun window is counted here as a gap (same as the R code).
    ge = oc[rd + ["PRIME_TIME_END", "last_case_end_tat_incl"]].drop_duplicates()
    ge = _with_cascade_window(ge)
    ge["gap_length"] = overlap_minutes(ge["cascade_start"], ge["cascade_end"],
                                       ge["last_case_end_tat_incl"], ge["PRIME_TIME_END"])
    ge = ge[ge["gap_length"].notna()].copy()
    ge["gap_length_normalized"] = ge["gap_length"] * ge["factor"]
    gaps_end = _label_sum(ge, rd, "gap_length_normalized", "End")

    # only room-days that have a between-case gap row are averaged (same as R: gaps_mid is the left table)
    gsr = (gaps_mid.merge(gaps_start, on=rd, how="left")
                   .merge(gaps_end, on=rd, how="left"))
    num = gsr.columns.difference(rd)
    gsr[num] = gsr[num].fillna(0)
    gsr["Recoverable"] = gsr["Recoverable End"] + gsr["Recoverable Mid"] + gsr["Recoverable Start"]
    gsr["Non Recoverable"] = gsr["Non Recoverable End"] + gsr["Non Recoverable Mid"] + gsr["Non Recoverable Start"]
    gsr["Year"] = gsr["SURGERY_DATE"].dt.year
    gaps_summary_room = gsr.groupby(["Location", "Year"], dropna=False).agg(**{
        "Prime Time Recoverable Time": ("Recoverable", "mean"),
        "Prime Time Non Recoverable Time": ("Non Recoverable", "mean")}).reset_index()

    # --- available time: prime-time window x staffed rooms in each cascade band ---
    av = base_rooms[["SURGERY_DATE", "Location", "PRIME_TIME_START", "PRIME_TIME_END"]].drop_duplicates()
    av = _with_cascade_window(av)
    av["minutes"] = overlap_minutes(av["cascade_start"], av["cascade_end"],
                                    av["PRIME_TIME_START"], av["PRIME_TIME_END"])
    av = av[av["minutes"].notna()].copy()
    av["avail"] = av["minutes"] * av["# ORs"]
    av["Year"] = av["SURGERY_DATE"].dt.year
    av_day = av.groupby(["Location", "SURGERY_DATE", "Year"], dropna=False)["avail"].sum().reset_index()
    rd_avail = (av_day.groupby(["Location", "Year"], dropna=False)["avail"].mean()
                      .rename("Prime Time Available Time").reset_index())

    out = (raw_all.merge(rd_avail, on=["Location", "Year"], how="left")
                  .merge(gaps_summary_room, on=["Location", "Year"], how="left"))
    fill = [c for c in METRIC_COLS if c != "Prime Time Utilization"]
    out[fill] = out[fill].fillna(0)
    avail = out["Prime Time Available Time"]
    out["Prime Time Utilization"] = (out["Prime Time Used Time"] / avail).where(avail > 0)
    out["scenario"] = scenario_label
    out = out.rename(columns={"Location": "Hospital"})
    return out[["Hospital", "Year"] + METRIC_COLS + ["scenario"]]

### Projected band
Baseline metrics plus the new-volume pool per Hospital x Year. New minutes are added to Used and
taken out of Recoverable first. Available time doesn't change. Dummy cases need no room.

In [ ]:
def project_with_volume(baseline_out, dummy_cases, scenario_label):
    d = prime_time_location(dummy_cases)
    d["proc_pt_min"] = overlap_minutes(d["PRIME_TIME_START"], d["PRIME_TIME_END"],
                                       d["PATIENT_IN_ROOM_DTTM"], d["PATIENT_OUT_ROOM_DTTM"])
    d["tat_pt_min"] = overlap_minutes(d["PRIME_TIME_START"], d["PRIME_TIME_END"],
                                      d["PATIENT_OUT_ROOM_DTTM"], d["PATIENT_OUT_AND_SETUP_CLEANUP_END"])
    d["used_pt_min"] = d["proc_pt_min"].fillna(0) + d["tat_pt_min"].fillna(0)
    d["Year"] = d["SURGERY_DATE"].dt.year
    d["Hospital"] = d["Location"]
    day = d.groupby(["Hospital", "Year", "SURGERY_DATE"], dropna=False).agg(
        cases_day=("OR_CASE_ID", "size"), used_day=("used_pt_min", "sum"),
        proc_day=("proc_pt_min", "sum"), tat_day=("tat_pt_min", "sum")).reset_index()
    pool = day.groupby(["Hospital", "Year"], dropna=False).agg(
        dummy_cases=("cases_day", "sum"), dummy_min=("used_day", "mean"),
        dummy_min_proc_time=("proc_day", "mean"), dummy_min_tat_time=("tat_day", "mean")).reset_index()

    out = baseline_out.drop(columns="scenario").merge(pool, on=["Hospital", "Year"], how="outer", sort=False)
    num = out.select_dtypes("number").columns
    out[num] = out[num].fillna(0)
    out["# Cases"] = out["# Cases"] + out["dummy_cases"]
    out["Prime Time Used Time"] = out["Prime Time Used Time"] + out["dummy_min"]
    out["Prime Time Recoverable Time"] = (out["Prime Time Recoverable Time"] - out["dummy_min"]).clip(lower=0)
    out["Prime Time Procedure Time"] = out["Prime Time Procedure Time"] + out["dummy_min_proc_time"]
    out["Prime Time TAT"] = out["Prime Time TAT"] + out["dummy_min_tat_time"]
    avail = out["Prime Time Available Time"]
    out["Prime Time Utilization"] = (out["Prime Time Used Time"] / avail).where(avail > 0)
    out["scenario"] = scenario_label
    return out[["Hospital", "Year"] + METRIC_COLS + ["scenario"]]

### Demand engine
- **Baseline**: rooms in use per hour = distinct rooms with a case touching that hour, averaged over surgery days.
- **New volume**: case minutes inside each hour / 60, averaged over surgery days (dummies have no room).

In [ ]:
def demand_baseline(baseline_or_data):
    df = baseline_or_data[["LOCATION_NAME", "OR_CASE_ID", "SURGERY_DATE", "PATIENT_IN_ROOM_DTTM",
                           "PATIENT_OUT_ROOM_DTTM", "ROOM_ID"]]
    df = explode_hours(df, "PATIENT_IN_ROOM_DTTM", "PATIENT_OUT_ROOM_DTTM")
    df["hour_surgery"] = df["hour_ts"].dt.hour
    df = df.drop(columns="hour_ts").drop_duplicates()
    df["Location"] = xwalk_loc(df["LOCATION_NAME"]).to_numpy()
    df["YearReporting"] = df["SURGERY_DATE"].dt.year
    daily = (df.groupby(["Location", "SURGERY_DATE", "hour_surgery", "YearReporting"], dropna=False)["ROOM_ID"]
               .nunique(dropna=False).rename("RoomsInUse").reset_index())
    yearly = (daily.groupby(["Location", "hour_surgery", "YearReporting"], dropna=False)["RoomsInUse"]
                   .mean().reset_index())
    yearly = yearly.merge(cascade_hourly_capacity(), on=["Location", "hour_surgery"], how="left")
    yearly["Scenario"] = "Baseline"
    return yearly


def demand_new_volume(new_volume_or_cases, scenario="Scenario"):
    df = explode_hours(new_volume_or_cases, "PATIENT_IN_ROOM_DTTM", "PATIENT_OUT_ROOM_DTTM")
    h_end = df["hour_ts"] + pd.Timedelta(hours=1)
    seg_end = df["PATIENT_OUT_ROOM_DTTM"].where(df["PATIENT_OUT_ROOM_DTTM"] < h_end, h_end)
    seg_start = df["PATIENT_IN_ROOM_DTTM"].where(df["PATIENT_IN_ROOM_DTTM"] > df["hour_ts"], df["hour_ts"])
    df["minutes_in_hour"] = (seg_end - seg_start).dt.total_seconds() / 60
    df["hour_surgery"] = df["hour_ts"].dt.hour
    df["Location"] = xwalk_loc(df["LOCATION_NAME"]).to_numpy()
    df["YearReporting"] = df["SURGERY_DATE"].dt.year
    daily = (df.groupby(["Location", "SURGERY_DATE", "hour_surgery", "YearReporting"], dropna=False)
               ["minutes_in_hour"].sum().rename("ORMinutesDemand").reset_index())
    yearly = (daily.groupby(["Location", "hour_surgery", "YearReporting"], dropna=False)["ORMinutesDemand"]
                   .mean().div(60).rename("RoomsInUse").reset_index())
    yearly = yearly.merge(cascade_hourly_capacity(), on=["Location", "hour_surgery"], how="left")
    yearly["Scenario"] = scenario
    return yearly

## 1. Volume projection data

In [ ]:
volume_path = os.path.join(file_location, "archive", VOLUME_FILE_NAME)
data_baseline_ip = pd.read_excel(volume_path, sheet_name="baseline")
data_volume_projections_ip = pd.read_excel(volume_path, sheet_name="scenario")

for col in ["SERVICE_DATE", "ADMIT_DT_SRC", "DSCH_DT_SRC"]:
    data_baseline_ip[col] = pd.to_datetime(data_baseline_ip[col])
for col in ["NEW_ADMIT_DT_SRC", "NEW_DSCH_DT_SRC"]:
    data_volume_projections_ip[col] = pd.to_datetime(data_volume_projections_ip[col])

min_admin_date = data_baseline_ip["SERVICE_DATE"].min().strftime("%Y-%m-%d")
max_discharge_date = data_baseline_ip["SERVICE_DATE"].max().strftime("%Y-%m-%d")

mrn_list_ip = (data_baseline_ip.assign(MSMRN=as_id_string(data_baseline_ip["MSMRN"]).str.strip())
               [["MSMRN", "ADMIT_DT_SRC", "DSCH_DT_SRC"]]
               .drop_duplicates()
               .astype({"MSMRN": "string"}))
print(min_admin_date, max_discharge_date, len(mrn_list_ip), "IP stays")

## 2. OR cases for the baseline window

In [ ]:
or_cases_baseline = get_or_data(sched_start_date=min_admin_date, sched_end_date=max_discharge_date,
                                status="Completed", mrn_list=mrn_list_ip)

# Cohort flag: case falls inside a baseline IP stay
or_cases_baseline["is_cohort"] = or_cases_baseline["ADMIT_DT_SRC"].notna()
or_cases_baseline["is_cohort"].value_counts()

## 3. Baseline metrics

In [ ]:
summary_metrics_baseline = summary_metrics_weighted(or_cases_baseline, scenario_label="Baseline")
summary_metrics_baseline

## 4. Parent templates
One index surgery per cohort patient. `surgery_offset` = day of stay the surgery happened on,
`time_of_day` = in-room clock time.

In [ ]:
pt = or_cases_baseline[or_cases_baseline["is_cohort"]].copy()
pt["admit_date"] = pt["ADMIT_DT_SRC"].dt.normalize()
pt["surgery_offset"] = (pt["PATIENT_IN_ROOM_DTTM"].dt.normalize() - pt["admit_date"]).dt.days
pt["time_of_day"] = pt["PATIENT_IN_ROOM_DTTM"] - pt["PATIENT_IN_ROOM_DTTM"].dt.normalize()

# index case per stay, then one template per MRN (earliest stay wins; change the rule here if needed)
parent_templates = (pt.sort_values(["PATIENT_MRN", "admit_date", "PATIENT_IN_ROOM_DTTM"],
                                   na_position="last", kind="mergesort")
                      .drop_duplicates(["PATIENT_MRN", "admit_date"], keep="first")
                      .rename(columns={"PATIENT_MRN": "parent_mrn"})
                      [["parent_mrn", "surgery_offset", "time_of_day",
                        "MINUTES_IN_ROOM_TO_OUT_ROOM", "TURNOVER_FROM_PRIOR_CASE",
                        "PRIMARY_SURGEON", "PRIMARY_SURGEON_SPECIALTY",
                        "PRIMARY_PROCEDURE_CODE", "PRIMARY_PROCEDURE_DESC",
                        "ANESTHESIA_TYPE", "LOCATION_NAME"]]
                      .drop_duplicates("parent_mrn", keep="first")
                      .reset_index(drop=True))
len(parent_templates)

## 5. Dummy cases for new volume
Each new encounter (MRN with a `_n` suffix) copies its parent's surgery, shifted to the new admit date.
Dates that land on a holiday or weekend are moved forward to a working day.

In [ ]:
nv = data_volume_projections_ip.copy()
nv["MSMRN"] = as_id_string(nv["MSMRN"]).str.strip()
nv = nv[nv["MSMRN"].str.contains("_", regex=False, na=False)].copy()
nv["NEW_ADMIT_DT_SRC"] = nv["NEW_ADMIT_DT_SRC"].dt.normalize()
nv["NEW_DSCH_DT_SRC"] = nv["NEW_DSCH_DT_SRC"].dt.normalize()
nv["parent_mrn"] = nv["MSMRN"].str.replace(r"_\d+$", "", regex=True)
nv = (nv[["ENCOUNTER_NO", "MSMRN", "parent_mrn", "NEW_ADMIT_DT_SRC", "NEW_DSCH_DT_SRC"]]
        .drop_duplicates()
        .astype({"parent_mrn": "string"})
        .merge(parent_templates, on="parent_mrn", how="inner"))

nv["SURGERY_DATE_OFFSET"] = nv["NEW_ADMIT_DT_SRC"] + pd.to_timedelta(nv["surgery_offset"], unit="D")
nv["SURGERY_DATE"] = get_valid_date(nv["SURGERY_DATE_OFFSET"]).to_numpy()
nv["PATIENT_IN_ROOM_DTTM"] = nv["SURGERY_DATE"] + nv["time_of_day"]
nv["PATIENT_OUT_ROOM_DTTM"] = add_minutes(nv["PATIENT_IN_ROOM_DTTM"], nv["MINUTES_IN_ROOM_TO_OUT_ROOM"])
nv["OR_CASE_ID"] = "NEW_" + as_id_string(nv["ENCOUNTER_NO"])
nv["PATIENT_MRN"] = nv["MSMRN"]
new_volume_cases = nv
print(len(new_volume_cases), "dummy cases;",
      int((new_volume_cases["SURGERY_DATE_OFFSET"] != new_volume_cases["SURGERY_DATE"]).sum()), "date-shifted")

## 6. Projected metrics
Dummies have no room, so they don't enter the gap engine. They add to the raw columns, and their minutes
come out of baseline recoverable time.

In [ ]:
new_volume_cases_raw = prime_time_location(new_volume_cases)
setup_end = add_minutes(new_volume_cases_raw["PATIENT_OUT_ROOM_DTTM"], new_volume_cases_raw["TURNOVER_FROM_PRIOR_CASE"])
new_volume_cases_raw["PATIENT_OUT_AND_SETUP_CLEANUP_END"] = setup_end.fillna(new_volume_cases_raw["PATIENT_OUT_ROOM_DTTM"])
new_volume_cases_raw["overlap_primetime_procedure"] = 1.0   # non-NA placeholders, as in the R code
new_volume_cases_raw["overlap_primetime_setup"] = 1.0
new_volume_cases_raw["Location"] = xwalk_loc(new_volume_cases_raw["LOCATION_NAME"]).to_numpy()

output_projected = project_with_volume(summary_metrics_baseline, new_volume_cases_raw, "Volume Projections")
output_all = pd.concat([summary_metrics_baseline, output_projected], ignore_index=True)
output_all

## 7. Delta table
All Baseline columns, then Volume Projections, then deltas.

In [ ]:
grain = ["Hospital", "Year"]
output_wide = output_all[grain].drop_duplicates().reset_index(drop=True)
for scen in output_all["scenario"].drop_duplicates():
    part = output_all.loc[output_all["scenario"] == scen, grain + METRIC_COLS]
    part = part.rename(columns={c: f"{scen} {c}" for c in METRIC_COLS})
    output_wide = output_wide.merge(part, on=grain, how="left")

base_cols_all = [c for c in output_wide.columns if c.startswith("Baseline ")]
output_wide[base_cols_all] = output_wide[base_cols_all].fillna(0)

vp, bl = "Volume Projections ", "Baseline "
output_wide["Delta Cases"] = output_wide[vp + "# Cases"] - output_wide[bl + "# Cases"]
output_wide["Delta Used"] = output_wide[vp + "Prime Time Used Time"] - output_wide[bl + "Prime Time Used Time"]
output_wide["Delta Procedure Time"] = output_wide[vp + "Prime Time Procedure Time"] - output_wide[bl + "Prime Time Procedure Time"]
# PARITY NOTE: the R code subtracts Volume Projections TAT from itself, so this is always 0.
# Change the second term to bl + "Prime Time TAT" to get the real delta.
output_wide["Delta TAT"] = output_wide[vp + "Prime Time TAT"] - output_wide[vp + "Prime Time TAT"]
output_wide["Delta Utilization"] = output_wide[vp + "Prime Time Utilization"] - output_wide[bl + "Prime Time Utilization"]
output_wide["Delta Recoverable"] = output_wide[vp + "Prime Time Recoverable Time"] - output_wide[bl + "Prime Time Recoverable Time"]
output_wide

## 8. Demand vs staffed capacity

In [ ]:
new_volume_cases_raw_subset = new_volume_cases_raw[["OR_CASE_ID", "SURGERY_DATE", "PATIENT_IN_ROOM_DTTM",
                                                    "PATIENT_OUT_ROOM_DTTM", "LOCATION_NAME"]]

baseline_demand = demand_baseline(or_cases_baseline)
new_volume_demand = demand_new_volume(new_volume_cases_raw_subset, scenario="Volume Projections")

demand_capacity_aggregated = pd.concat([baseline_demand, new_volume_demand], ignore_index=True)
yr = demand_capacity_aggregated["YearReporting"]
demand_capacity_aggregated = demand_capacity_aggregated[yr.notna() & (yr != EXCLUDE_YEAR)].reset_index(drop=True)
demand_capacity_aggregated.head()

### Plots: OR staffed capacity vs demand, one per site

In [ ]:
BAR_W = 0.7
HOURS = list(range(7, 20))
SCENARIO_COLORS = {"Baseline": mshs_violet, "Volume Projections": mshs_cyan}

plt.rcParams.update({"font.family": "DejaVu Sans"})


def plot_site(site, data):
    sd = data[(data["Location"] == site) & data["hour_surgery"].isin(HOURS)]
    site_data = (sd.groupby(["hour_surgery", "Scenario"])
                   .agg(RoomsInUse=("RoomsInUse", "sum"), Capacity=("Capacity", "max"))
                   .reset_index())
    cap_data = site_data.groupby("hour_surgery")["Capacity"].max().dropna()

    fig, ax = plt.subplots(figsize=(12, 6))
    bottoms = pd.Series(0.0, index=HOURS)
    for scen in ["Baseline", "Volume Projections"]:            # Baseline at the bottom, new volume on top
        s = site_data[site_data["Scenario"] == scen].set_index("hour_surgery")["RoomsInUse"]
        if s.empty:
            continue
        b = bottoms.reindex(s.index)
        ax.bar(s.index, s.values, width=BAR_W, bottom=b.values, color=SCENARIO_COLORS[scen], zorder=2)
        for h, v, b0 in zip(s.index, s.values, b.values):
            ax.text(h, b0 + v / 2, f"{round(v, 1):g}", ha="center", va="center",
                    fontsize=6.8, fontweight="bold", color="white", zorder=3)
        bottoms.loc[s.index] += s.values

    for h, c in cap_data.items():
        ax.hlines(c, h - 0.5, h + 0.5, color=mshs_magenta, linewidth=2, zorder=4)

    top = max(bottoms.max(), cap_data.max() if not cap_data.empty else 0)
    ax.set_ylim(0, top * 1.1 if top > 0 else 1)
    ax.set_xlim(HOURS[0] - 0.6, HOURS[-1] + 0.6)
    ax.set_xticks(HOURS)
    ax.tick_params(axis="both", labelsize=10, length=0)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(f"OR Staffed Capacity Vs Demand (Location: {site})", fontsize=14, fontweight="bold")
    ax.set_xlabel("Time of Day (24h Format)", fontsize=12)
    ax.set_ylabel("Room Demand", fontsize=12)

    fill_legend = ax.legend(handles=[Patch(color=SCENARIO_COLORS[k], label=k) for k in SCENARIO_COLORS],
                            title="Demand Scenario", loc="upper left", bbox_to_anchor=(1.02, 0.6),
                            frameon=False, fontsize=8, title_fontsize=12, alignment="left")
    ax.add_artist(fill_legend)
    ax.legend(handles=[Line2D([0], [0], color=mshs_magenta, linewidth=2, label="Staffed Capacity")],
              title="Threshold", loc="upper left", bbox_to_anchor=(1.02, 0.38),
              frameon=False, fontsize=8, title_fontsize=12, alignment="left")
    fig.tight_layout(rect=(0, 0, 0.84, 1))   # leave room on the right for the legends
    return fig


for site in demand_capacity_aggregated["Location"].dropna().unique():
    fig = plot_site(site, demand_capacity_aggregated)
    fig.savefig(os.path.join(plot_dir, f"demand_vs_staffed_{PLOT_VERSION}_{site}.png"), dpi=150)
    plt.show()
    plt.close(fig)

## 9. Write the results workbook
"Output by Grain" gets a two-row header: row 1 merges Baseline / Volume Projections / Delta across their
column groups, and row 2 has the detail column names.

In [ ]:
def strip_band(names):
    out = []
    for n in names:
        for prefix in ("Baseline ", "Volume Projections "):
            if n.startswith(prefix):
                n = n[len(prefix):]
        if n.startswith("Delta"):
            n = n[len("Delta"):].lstrip(" ")
        out.append(n)
    return out


grain_cols = ["Hospital", "Year"]
base_cols = [c for c in output_wide.columns if c.startswith("Baseline ")]
proj_cols = [c for c in output_wide.columns if c.startswith("Volume Projections ")]
delta_cols = [c for c in output_wide.columns if c.startswith("Delta")]
ordered_cols = grain_cols + base_cols + proj_cols + delta_cols
output_wide = output_wide[ordered_cols]

wb = Workbook()
ws = wb.active
ws.title = "Output by Grain"
n_grain, n_base, n_proj, n_delta = len(grain_cols), len(base_cols), len(proj_cols), len(delta_cols)

# Row 1: band labels, merged across each group
bands = [("Baseline", n_grain + 1, n_base), ("Volume Projections", n_grain + n_base + 1, n_proj),
         ("Delta", n_grain + n_base + n_proj + 1, n_delta)]
for label, start_col, width in bands:
    if width > 0:
        ws.cell(row=1, column=start_col, value=label)
        if width > 1:
            ws.merge_cells(start_row=1, start_column=start_col, end_row=1, end_column=start_col + width - 1)

# Row 2: detail headers
for j, name in enumerate(grain_cols + strip_band(base_cols + proj_cols + delta_cols), start=1):
    ws.cell(row=2, column=j, value=name)

# Data from row 3
for i, row in enumerate(output_wide.itertuples(index=False), start=3):
    for j, v in enumerate(row, start=1):
        if isinstance(v, (float, np.floating)) and np.isnan(v):
            v = None
        elif isinstance(v, np.generic):
            v = v.item()
        ws.cell(row=i, column=j, value=v)

thin = Side(style="thin")
band_fill = PatternFill("solid", fgColor="D9E1F2")
for j in range(1, len(ordered_cols) + 1):
    c1 = ws.cell(row=1, column=j)
    c1.font = Font(bold=True)
    c1.alignment = Alignment(horizontal="center")
    c1.fill = band_fill
    c1.border = Border(left=thin, right=thin, top=thin, bottom=thin)
    c2 = ws.cell(row=2, column=j)
    c2.font = Font(bold=True)
    c2.alignment = Alignment(horizontal="center", wrap_text=True)

ws.freeze_panes = "A3"
for j in range(1, len(ordered_cols) + 1):
    width = max(len(str(c.value)) for c in ws[get_column_letter(j)][1:] if c.value is not None) if ws.max_row > 1 else 10
    ws.column_dimensions[get_column_letter(j)].width = min(max(width + 2, 8), 60)

out_file = os.path.join(output_dir, f"or_capacity_scenario_results {datetime.now():%Y-%m-%d %H:%M:%S}.xlsx")
wb.save(out_file)
print("Saved", out_file)

## 10. Validation
MSH, average per surgery day around 11:00: cases touching hour 11, cases in the room at 11:00, and distinct
rooms in use at 11:00 (occupancy includes turnover).

The first validation block in `or_model.R` uses `baseline_collision` and `all_cases`, which the script
never creates (they came from the old collision code), so it isn't ported.

In [ ]:
v = or_cases_baseline.copy()
v["casc_loc"] = xwalk_loc(v["LOCATION_NAME"]).to_numpy()
v["OCC_END"] = add_minutes(v["PATIENT_OUT_ROOM_DTTM"], v["TURNOVER_FROM_PRIOR_CASE"].fillna(0))
v["in_s"] = (v["PATIENT_IN_ROOM_DTTM"] - v["PATIENT_IN_ROOM_DTTM"].dt.normalize()).dt.total_seconds()
v["out_s"] = (v["OCC_END"] - v["OCC_END"].dt.normalize()).dt.total_seconds()
v = v[v["casc_loc"] == "MSH"]
v["touching"] = (v["in_s"] < 12 * 3600) & (v["out_s"] > 11 * 3600)
v["occupied"] = (v["in_s"] <= 11 * 3600) & (v["out_s"] > 11 * 3600)
per_day = v.groupby("SURGERY_DATE").apply(lambda d: pd.Series({
    "touching_hr11": d["touching"].sum(),
    "occupied_at_1100": d["occupied"].sum(),
    "distinct_rooms_at_1100": d.loc[d["occupied"], "ROOM_ID"].nunique(dropna=False),
}))
per_day.mean().to_frame("mean per day")